In [ ]:
!pip install git+https://github.com/huggingface/parler-tts.git transformers scipy openai-whisper jiwer

  Cloning https://github.com/huggingface/parler-tts.git to /tmp/pip-req-build-kstk9j1z
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/parler-tts.git /tmp/pip-req-build-kstk9j1z
  Resolved https://github.com/huggingface/parler-tts.git to commit d108732cd57788ec86bc857d99a6cabd66663d68
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 25.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Cloning https://github.com/descriptinc/audiotools to /tmp/pip-install-ymdcr6jn/descript-audiotools_2ffc7cccf02742f69e15e6e3de29623f
  Running command git clone --filter=blob:none --quiet https://github.com/descriptinc/audiotools /tmp/pip-install-ymdcr6jn/descript-audiotools_2ffc7cccf02742f69e15e6e3de29623f
  Resolved http

In [ ]:
from huggingface_hub import login

login(token="YOUR_HF_TOKEN_HERE")

In [ ]:
!pip install --upgrade protobuf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 12.2 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 4.25.9
    Uninstalling protobuf-4.25.9:
      Successfully uninstalled protobuf-4.25.9
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
descript-audiotools 0.7.4 requires protobuf!=4.24.0,<5.0.0,>=3.19.6, but you have protobuf 7.35.1 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 7.35.1 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 7.35.1 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.35.1 which is incompatible.


In [ ]:
import json
import os
import shutil
import torch
import scipy.io.wavfile
import librosa
from transformers import AutoTokenizer, WhisperProcessor, WhisperForConditionalGeneration
from parler_tts import ParlerTTSForConditionalGeneration
from jiwer import wer, cer
from IPython.display import Audio, display

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading AI4Bharat Indic Parler-TTS model...")
# Initialize the TTS Model
tts_model_id = "ai4bharat/indic-parler-tts"
tts_model = ParlerTTSForConditionalGeneration.from_pretrained(tts_model_id).to(device)
tts_tokenizer = AutoTokenizer.from_pretrained(tts_model_id)

# 1. Define the Voice Characteristic (Caption)
# Parler-TTS uses this to shape the acoustic output.
voice_description = "A female speaker delivers a clear and expressive speech in Nepali at a moderate pace. The recording is of very high quality."
input_ids = tts_tokenizer(voice_description, return_tensors="pt").input_ids.to(device)

print("Loading Whisper Medium ASR model...")
# Initialize the Evaluation ASR Model
asr_model_id = "openai/whisper-medium"
asr_processor = WhisperProcessor.from_pretrained(asr_model_id)
asr_model = WhisperForConditionalGeneration.from_pretrained(asr_model_id).to(device)

forced_decoder_ids = asr_processor.get_decoder_prompt_ids(language="nepali", task="transcribe")

# 2. Load Dataset
json_filename = "nepali_evaluation_set.json"
try:
    with open(json_filename, "r", encoding="utf-8") as f:
        eval_dataset = json.load(f)
    print(f"\nSuccessfully loaded {len(eval_dataset)} test sentences.")
except FileNotFoundError:
    print(f"Error: Could not find '{json_filename}'. Please upload it to Colab.")
    eval_dataset = []

# 3. Setup Output Directory
output_dir = "nepali_parler_tts_audio"
os.makedirs(output_dir, exist_ok=True)

print("\nStarting Generation and Evaluation...\n" + "="*60)

# 4. Evaluation Loop
for item in eval_dataset:
    item_id = item.get("ID", "Unknown_ID")
    ref_text = item.get("TEXT", "")
    category = item.get("EVALUATION SET", "Unknown_Category")

    # --- A. Generate Audio (TTS) ---
    # The transcript goes into prompt_input_ids for Parler models
    prompt_input_ids = tts_tokenizer(ref_text, return_tensors="pt").input_ids.to(device)

    with torch.no_grad():
        generation = tts_model.generate(input_ids=input_ids, prompt_input_ids=prompt_input_ids)

    # Extract audio array and save
    audio_data = generation.cpu().numpy().squeeze()
    file_path = os.path.join(output_dir, f"{item_id}.wav")
    scipy.io.wavfile.write(file_path, tts_model.config.sampling_rate, audio_data)

    # --- B. Transcribe Audio (ASR) ---
    speech_array, sampling_rate = librosa.load(file_path, sr=16000)
    input_features = asr_processor(speech_array, sampling_rate=16000, return_tensors="pt").input_features.to(device)

    with torch.no_grad():
        predicted_ids = asr_model.generate(input_features, forced_decoder_ids=forced_decoder_ids)

    hyp_text = asr_processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

    # --- C. Calculate Error Rates ---
    # Normalizing text
    norm_ref = ref_text.replace("।", "").replace(",", "").replace("?", "").replace("!", "").strip()
    norm_hyp = hyp_text.replace("।", "").replace(",", "").replace("?", "").replace("!", "").strip()

    try:
        item_wer = wer(norm_ref, norm_hyp)
        item_cer = cer(norm_ref, norm_hyp)
    except ValueError:
        item_wer, item_cer = 1.0, 1.0

    print(f"ID {item_id} | {category}")
    print(f"  Reference  : {norm_ref}")
    print(f"  Prediction : {norm_hyp}")
    print(f"  --> WER: {item_wer:.4f} ({item_wer * 100:.1f}%) | CER: {item_cer:.4f} ({item_cer * 100:.1f}%)\n")
    print("-" * 60)

# 5. Zip and Download the Audio Folder
if os.path.exists(output_dir) and len(os.listdir(output_dir)) > 0:
    print(f"\nZipping the '{output_dir}' directory...")
    zip_filename = f"{output_dir}.zip"
    shutil.make_archive(output_dir, 'zip', output_dir)
    print(f"Archive saved as {zip_filename}. Ready for download.")

Loading AI4Bharat Indic Parler-TTS model...


  "_name_or_path": "google/flan-t5-large",
  "architectures": [
    "T5ForConditionalGeneration"
  ],
  "classifier_dropout": 0.0,
  "d_ff": 2816,
  "d_kv": 64,
  "d_model": 1024,
  "decoder_start_token_id": 0,
  "dense_act_fn": "gelu_new",
  "dropout_rate": 0.1,
  "eos_token_id": 1,
  "feed_forward_proj": "gated-gelu",
  "initializer_factor": 1.0,
  "is_encoder_decoder": true,
  "is_gated_act": true,
  "layer_norm_epsilon": 1e-06,
  "model_type": "t5",
  "n_positions": 512,
  "num_decoder_layers": 24,
  "num_heads": 16,
  "num_layers": 24,
  "output_past": true,
  "pad_token_id": 0,
  "relative_attention_max_distance": 128,
  "relative_attention_num_buckets": 32,
  "tie_word_embeddings": false,
  "transformers_version": "4.46.1",
  "use_cache": true,
  "vocab_size": 32128
}

  "_name_or_path": "ylacombe/dac_44khz",
  "architectures": [
    "DacModel"
  ],
  "codebook_dim": 8,
  "codebook_loss_weight": 1.0,
  "codebook_size": 1024,
  "commitment_loss_weight": 0.25,
  "decoder_hidden_si

Loading Whisper Medium ASR model...


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



Successfully loaded 20 test sentences.

Starting Generation and Evaluation...
ID NEP_01 | velars_gutturals
  Reference  : कागती खाएर केटाकेटीहरू खुसी हुँदै घर गए
  Prediction : कारित्ति खायरब् केटुकिट्यों खुसी हुने येरते ने
  --> WER: 0.8571 (85.7%) | CER: 0.5897 (59.0%)

------------------------------------------------------------
ID NEP_02 | palatals_and_trills
  Reference  : चराहरू चिरबिर गर्दै चाँडै उडेर चौतारीमा बसे
  Prediction : चारा आरूची रुबिर करता ही चणणे वोड़े शुवाद्री मार्मुसे
  --> WER: 1.2857 (128.6%) | CER: 0.7209 (72.1%)

------------------------------------------------------------
ID NEP_03 | retroflexes_and_nasalization
  Reference  : ठूलो डाँडामाथि ढकमक्क गुराँस फुल्दा धेरै राम्रो देखिन्छ
  Prediction : अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अपने अप